In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
import statsmodels.api as sm
from scipy.stats import chi2
from statsmodels.tsa.api import VAR


In [ ]:
# Question 1

# Load data

data = pd.read_excel('Assignment2Data.xlsx', sheet_name='FamaFrenchSize')

# Create datetime and set as index
data['Date'] = pd.to_datetime(data[['Year', 'Month']].assign(DAY = 1))
data.set_index('Date', inplace=True)

output_file = 'Assignment2_results.xlsx'

# Question 1a)

# Recalculate the returns and create ln returns
data['Small'] = data['Small'] / 100
data['Medium'] = data['Medium'] / 100
data['Big'] = data['Big'] / 100

data['Small_log_ret'] = np.log(1 + data['Small'])
data['Medium_log_ret'] = np.log(1 + data['Medium'])
data['Big_log_ret'] = np.log(1 + data['Big'])


In [ ]:
# Question 1a + b)
# Setup
portfolios = ['Small', 'Medium', 'Big']
results = {portfolio: {'horizon': [], 'beta': [], 'beta t-value': [],'beta p-value': [], 'HH se': [],
                       'NW se': [], 'HH t-value': [], 'NW t-value': [],} for portfolio in ['Small', 'Medium', 'Big']}
horizons = [1, 12, 24, 36, 48, 60, 72, 96, 120]

# Run regressions for each horizon and each portfolio
for K in horizons:
    for portfolio in portfolios:
        # Prepare data for regression
        r_future = data[f'{portfolio}_log_ret'].rolling(window=K).sum().shift(-K)
        r_past = data[f'{portfolio}_log_ret'].rolling(window=K).sum()
        
        # Drop NaNs
        df = pd.DataFrame({'r_future': r_future, 'r_past': r_past}).dropna()
        
        # Run regression
        X = add_constant(df['r_past'])
        y = df['r_future']
        model = OLS(y, X).fit()

        # Hansen and Hodrick standard errors
        hh_se = model.get_robustcov_results(cov_type='HAC', maxlags=K-1, use_correction=False)
        
        # Newey and West standard errors
        nw_se = model.get_robustcov_results(cov_type='HAC', maxlags=K-1, use_correction=True)

        # Print results
        print(f'Portfolio: {portfolio}, Horizon: {K} months')
        print(f'Slope Coefficient: {model.params["r_past"]:.4f}')
        print(f'Slope p-value: {model.pvalues["r_past"]:.4f}')
        print(f'Hansen-Hodrick SE: {hh_se.bse[1]:.4f}')
        print(f'Newey-West SE: {nw_se.bse[1]:.4f}')
        print(f'Hansen-Hodrick t-Statistic: {hh_se.tvalues[1]:.4f}')
        print(f'Newey-West t-Statistic: {nw_se.tvalues[1]:.4f}')
        print('------------------------------------------')
        
        # Store results
        results[portfolio]['horizon'].append(K)
        results[portfolio]['beta'].append(model.params['r_past'])
        results[portfolio]['beta p-value'].append(model.pvalues['r_past'])
        results[portfolio]['HH se'].append(hh_se.bse[1])
        results[portfolio]['HH t-value'].append(hh_se.tvalues[1])
        results[portfolio]['NW se'].append(nw_se.bse[1])
        results[portfolio]['NW t-value'].append(nw_se.tvalues[1])

# Save to excel

rows = []
for portfolio in portfolios:
    for i, horizon in enumerate(results[portfolio]['horizon']):
        rows.append({
            'Portfolio': portfolio,
            'Horizon': horizon,
            'Beta': results[portfolio]['beta'][i],
            'Beta P-Value': results[portfolio]['beta p-value'][i],
            'Hansen-Hodrick SE': results[portfolio]['HH se'][i],
            'Hansen-Hodrick T-Value': results[portfolio]['HH t-value'][i],
            'Newey-West SE': results[portfolio]['NW se'][i],
            'Newey-West T-Value': results[portfolio]['NW t-value'][i]
        })

results_df = pd.DataFrame(rows)

with pd.ExcelWriter(output_file, engine='openpyxl', mode='a' if os.path.exists(output_file) else 'w', if_sheet_exists='replace') as writer:
    results_df.to_excel(writer, sheet_name='Question1', index=False)


# Plot
plt.figure(figsize=(10, 6))
for portfolio in portfolios:
    plt.plot(results[portfolio]['horizon'], results[portfolio]['beta'], marker='o', label=portfolio)
plt.title('Slope Coefficients vs Horizon')
plt.xlabel('Horizon (Months)')
plt.ylabel('Slope Coefficient (β)')
plt.legend()
plt.grid(True)
plt.savefig('Assignment2_1a_beta_plot.jpg', format = 'jpg', dpi = 300)
plt.show()

In [ ]:
# 1c)
# To test the combined significance of all slopes we can perform the Wald test
# We will use the Newey-West covariance matrix in order to get estimates robust to heteroskedasticity and autocorrelation

for portfolio in portfolios:
    beta_hat = np.array(results[portfolio]['beta'])
    se_hat = np.array(results[portfolio]['NW se'])
    t_stats = beta_hat / se_hat

    # Wald statistic (sum of squared t-stats)
    W = np.sum(t_stats ** 2)
    df = len(beta_hat)  

    p_value = 1 - chi2.cdf(W, df)

    print(f'\nJoint Significance Test for {portfolio}:')
    print(f'Wald Statistic: {W:.4f}')
    print(f'Degrees of Freedom: {df}')
    print(f'p-value: {p_value:.4f}')
    if p_value < 0.05:
        print('Reject the null hypothesis: At least one slope coefficient is significant.')
    else:
        print('Fail to reject the null hypothesis: Slopes are jointly not significant.')


# Question 2

In [ ]:
# Question 2

# Load data
output_file = 'Assignment2_results.xlsx'
data = pd.read_excel('Assignment2Data.xlsx', sheet_name='CRSP')

data['Date'] = pd.to_datetime(dict(year=data['Year'], month=data['Month'], day=1))
data.set_index('Date', inplace=True)
data['Year'] = data.index.year

# Convert to decimals
data['Stock Return w/ Dividends'] = data['Stock return w/ dividends'] / 100
data['Stock Return w/o Dividends'] = data['Stock return w/o dividends'] / 100
data['T-bill Return'] = data['T-bill return'] / 100

# Dividend
data['Dividend'] = data['Stock Return w/ Dividends'] - data['Stock Return w/o Dividends']

# Function to calculate dividends with either cash or market reinvestment
def compute_annual_dividends_cash(group):
    cash_account = 0.0
    stock_investment = 1.0
    for idx, row in group.iterrows():
        dividend_yield = row['Dividend']
        t_bill_return = row['T-bill Return']
        # Dividend received for the month
        dividend = dividend_yield * stock_investment
        # Update cash account with T-bill returns and new dividend
        cash_account = cash_account * (1 + t_bill_return) + dividend
    return cash_account

def compute_annual_dividends_market(group):
    # Initialize variables
    stock_investment_with_div = 1.0
    stock_investment_without_div = 1.0
    for idx, row in group.iterrows():
        stock_return_with_div = row['Stock Return w/ Dividends']
        stock_return_without_div = row['Stock Return w/o Dividends']
        # Update investments
        stock_investment_with_div *= (1 + stock_return_with_div)
        stock_investment_without_div *= (1 + stock_return_without_div)
    # Total dividends are the difference between the two investments
    dividends = stock_investment_with_div - stock_investment_without_div
    return dividends

# Function to calculate total returns
def compute_annual_return(group, return_column):
    return np.prod(1 + group[return_column].values) - 1

# Dividends with cash reinvestment
annual_dividends_cash = data.groupby('Year').apply(compute_annual_dividends_cash)
annual_dividends_market = data.groupby('Year').apply(compute_annual_dividends_market)

# Total return
annual_total_return = data.groupby('Year').apply(
    compute_annual_return, return_column='Stock Return w/ Dividends')

# Compute annual price returns (without dividends)
annual_price_return = data.groupby('Year').apply(
    compute_annual_return, return_column='Stock Return w/o Dividends'
)

# Compute cumulative annual price
annual_price = (1 + annual_price_return).cumprod()

# Dividend growth rates
dividend_growth_cash = annual_dividends_cash.pct_change()
dividend_growth_market = annual_dividends_market.pct_change()


dividend_price_ratio_cash = annual_dividends_cash / annual_price
dividend_price_ratio_market = annual_dividends_market / annual_price

# Log returns
log_returns = np.log(1 + annual_total_return)

# Log dividend growth rates
log_dividend_growth_cash = np.log(annual_dividends_cash) - np.log(annual_dividends_cash.shift(1))
log_dividend_growth_market = np.log(annual_dividends_market) - np.log(annual_dividends_market.shift(1))

# Log dividend-price ratios
log_dividend_price_ratio_cash = np.log(annual_dividends_cash) - np.log(annual_price)
log_dividend_price_ratio_market = np.log(annual_dividends_market) - np.log(annual_price)

# Plot dividend growth rates
plt.figure(figsize=(12, 6))
plt.plot(log_dividend_growth_cash.index, log_dividend_growth_cash, label='Cash-Invested Dividends')
plt.plot(log_dividend_growth_market.index, log_dividend_growth_market, label='Market-Invested Dividends')
plt.title('Dividend Growth Rates (Log)')
plt.xlabel('Year')
plt.ylabel('Log Dividend Growth Rate')
plt.legend()
plt.savefig('A2_dividend_growth.jpg', format='jpg', dpi=300)
plt.show()

# Plot dividend-price ratios
plt.figure(figsize=(12, 6))
plt.plot(log_dividend_price_ratio_cash.index, log_dividend_price_ratio_cash, label='Cash-Invested Dividends')
plt.plot(log_dividend_price_ratio_market.index, log_dividend_price_ratio_market, label='Market-Invested Dividends')
plt.title('Dividend-Price Ratios (Log)')
plt.xlabel('Year')
plt.ylabel('Log Dividend-Price Ratio')
plt.legend()
plt.savefig('A2_dividend_price_ratio.jpg', format='jpg', dpi=300)
plt.show()

# Mean and Standard Deviation for Cash-Invested Dividends
mean_cash = log_dividend_growth_cash.mean()
std_cash = log_dividend_growth_cash.std()

# Mean and Standard Deviation for Market-Invested Dividends
mean_market = log_dividend_growth_market.mean()
std_market = log_dividend_growth_market.std()

print('Cash-Invested Dividends Dividend Growth Rate:')
print(f'Mean: {mean_cash:.4f}, Standard Deviation: {std_cash:.4f}')

print('Market-Invested Dividends Dividend Growth Rate:')
print(f'Mean: {mean_market:.4f}, Standard Deviation: {std_market:.4f}')


In [ ]:
print(dividend_price_ratio_cash)

In [ ]:
# 2b)
# Setup data
df = pd.DataFrame({
    'Year': annual_total_return.index,
    'Log_Returns': log_returns.values,
    'Log_Dividend_Growth': log_dividend_growth_cash.values,
    'Log_DP': log_dividend_price_ratio_cash.values
})

# Create the lagged log dividend-price ratio
df['Lag_Log_DP'] = df['Log_DP'].shift(1)

df.dropna(inplace=True)

def run_regression(df, dependent_var, independent_var):
    X = sm.add_constant(df[independent_var])
    y = df[dependent_var]
    model = sm.OLS(y, X).fit()
    return model

# Run regression of Log_Returns on Lag_Log_DP
model_returns_full = run_regression(df, 'Log_Returns', 'Lag_Log_DP')

# Run regression of Log_Dividend_Growth on Lag_Log_DP
model_div_growth_full = run_regression(df, 'Log_Dividend_Growth', 'Lag_Log_DP')

# Sub-sample up to 1989 (inclusive)
df_pre_1990 = df[df['Year'] <= 1989]

# Sub-sample from 1990 onwards
df_post_1990 = df[df['Year'] >= 1990]

# Pre-1990
model_returns_pre1990 = run_regression(df_pre_1990, 'Log_Returns', 'Lag_Log_DP')
model_div_growth_pre1990 = run_regression(df_pre_1990, 'Log_Dividend_Growth', 'Lag_Log_DP')

# Post-1990
model_returns_post1990 = run_regression(df_post_1990, 'Log_Returns', 'Lag_Log_DP')
model_div_growth_post1990 = run_regression(df_post_1990, 'Log_Dividend_Growth', 'Lag_Log_DP')

def report_regression_results(model, description):
    coef = model.params['Lag_Log_DP']
    t_stat = model.tvalues['Lag_Log_DP']
    p_value = model.pvalues['Lag_Log_DP']
    r_squared = model.rsquared
    print(f'{description}')
    print(f'Coefficient: {coef:.4f}')
    print(f'T-statistic: {t_stat:.2f}')
    print(f'P-value: {p_value:.4f}')
    print(f'R-squared: {r_squared:.4f}')
    print('----------------------------------------')

# Full sample
report_regression_results(model_returns_full, 'Full Sample - Predicting Log Returns')
report_regression_results(model_div_growth_full, 'Full Sample - Predicting Log Dividend Growth')

# Pre-1990
report_regression_results(model_returns_pre1990, 'Pre-1990 - Predicting Log Returns')
report_regression_results(model_div_growth_pre1990, 'Pre-1990 - Predicting Log Dividend Growth')

# Post-1990
report_regression_results(model_returns_post1990, 'Post-1990 - Predicting Log Returns')
report_regression_results(model_div_growth_post1990, 'Post-1990 - Predicting Log Dividend Growth')



In [ ]:
# 2c)

# Unfortunately unsuccessful